# FastWAM - LIBERO world-action model (interactive + imagination)

Standalone demo of **FastWAM** - a *world-action* model built on **Wan2.2-TI2V-5B** (a T5 text
encoder, a Wan video VAE, and a video+action diffusion transformer) - driving the **LIBERO**
simulator, rendered **inline, right here in the notebook**. On first use, the setup cell downloads
the released model and replay data from their public Hugging Face repositories; later runs reuse
the persistent cache.

Two things happen below, both on the **fast route** our AMD Strix Halo port ships:

1. **Interactive simulator (fast action route).** Exactly like the MolmoAct2 interactive demo, but
   driven by FastWAM. From the current camera view + robot state + your instruction, the model
   *plans a short action chunk* (`infer_action`), the simulator executes it, then it re-plans - the
   same receding-horizon closed loop as the shipped LIBERO eval. Type an instruction, press
   **Send**, and watch the arm act; the status line shows the live per-plan latency.

2. **Video imagination - what the model \"dreams\".** FastWAM is a *world* model, so on its joint
   route (`infer_joint`) it can roll the future **video** forward as well as the actions. We imagine
   **two full episodes from two different tasks** as receding-horizon rollouts and show **ground
   truth on the left, FastWAM's imagined future on the right** so you can compare.

The sim's small web UI is served on `RT_PORT` (8080 by default); Ryzers runs the container
with `--network=host`, so it is reachable at <http://localhost:8080/> with nothing to
forward. FastWAM runs in its own isolated environment
(`/opt/fastwam-venv`), kept separate from the MolmoAct2 training environment.

**First-run download:** about 25 GB of model files plus a 1.35 GB replay archive. **Knobs (env):**
`SUITE=libero_object`, `TASK_ID=0`, `EPISODES=47,273`, `NUM_STEPS=20`, `RT_PORT=8080`.

## Goals

* Reproduce the FastWAM LIBERO demo from public model and dataset releases, with no workshop bundle
* Drive the simulator on the same fast receding-horizon action route the shipped eval uses
* See what makes a **world** model different from a plain policy: on its joint route it can roll the *future video* forward, not just predict the next actions
* Compare the model's **imagined** future against ground truth, side by side, over full episodes
* Run it all inline over the container's host network in an isolated ROCm venv, with no extra port to forward

In [ ]:
import os
import subprocess
import json
import sys
import time
import urllib.request

# Quiet ROCm/torch spam in any child process we spawn.
os.environ.setdefault("TORCH_BLAS_PREFER_HIPBLASLT", "0")
os.environ.setdefault("PYTHONWARNINGS", "ignore")

OUT_DIR = os.environ.get("OUT_DIR", "/ryzers/outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# FastWAM runs in its OWN isolated venv (the numpy-1.26.4 LIBERO sim stack + ROCm torch), separate
# from the MolmoAct2 train-venv, and on its OWN LIBERO config path so the two never clash.
FASTWAM_PY = os.path.join(os.environ.get("FASTWAM_VENV", "/opt/fastwam-venv"), "bin", "python")
LIBERO_CONFIG_PATH_FASTWAM = os.environ.get("LIBERO_CONFIG_PATH_FASTWAM", "/opt/libero-config-fastwam")

# Public downloads live under persistent HOME by default. An optional offline workshop image can
# override these paths to its read-only /opt tree through the same environment variables.
FASTWAM_CACHE = os.path.expanduser(os.environ.get("FASTWAM_CACHE", "~/.cache/fastwam"))
RELEASE_DIR = os.environ.get("FASTWAM_RELEASE_DIR", os.path.join(FASTWAM_CACHE, "fastwam_release"))
DATA_DIR = os.environ.get("FASTWAM_DATA_DIR", os.path.join(FASTWAM_CACHE, "data"))
DIFFSYNTH = os.environ.get("DIFFSYNTH_MODEL_BASE_PATH", os.path.join(FASTWAM_CACHE, "diffsynth"))
DOWNLOAD_DATASET = os.environ.get("FASTWAM_DOWNLOAD_DATASET", "1") == "1"

_SCRIPTS = "/ryzers/notebooks/scripts"
if _SCRIPTS not in sys.path:
    sys.path.insert(0, _SCRIPTS)
from ft_helpers import prepare_fastwam_assets

# Idempotently fetch the released checkpoint, the Wan T5/VAE/tokenizer components, and (for the
# imagination section) the public LIBERO-Object replay archive. Set HF_TOKEN for faster downloads.
prepare_fastwam_assets(RELEASE_DIR, DIFFSYNTH, DATA_DIR, include_dataset=DOWNLOAD_DATASET)

CKPT = os.environ.get("CKPT") or os.path.join(RELEASE_DIR, "libero_uncond_2cam224.pt")
DATASET_STATS = os.environ.get("DATASET_STATS") or os.path.join(
    RELEASE_DIR, "libero_uncond_2cam224_dataset_stats.json")


# Subprocesses (the sim server / videogen) get a clean env: drop the notebook's inline matplotlib
# backend, force plain hipBLAS + unbuffered output, and select the model-agnostic policy adapter.
def child_env(**extra):
    e = dict(os.environ)
    e.pop("MPLBACKEND", None)
    e.update({
        "TORCH_BLAS_PREFER_HIPBLASLT": "0", "PYTHONWARNINGS": "ignore", "PYTHONUNBUFFERED": "1",
        "LIBERO_CONFIG_PATH": LIBERO_CONFIG_PATH_FASTWAM,
        "POLICY_FACTORY": "fastwam_libero_policy:build_policy",
        "CKPT": CKPT, "DATASET_STATS": DATASET_STATS,
        "DIFFSYNTH_MODEL_BASE_PATH": DIFFSYNTH,
        "DIFFSYNTH_DOWNLOAD_SOURCE": "huggingface",
        "FASTWAM_RELEASE_DIR": RELEASE_DIR,
        "FASTWAM_DATA_DIR": DATA_DIR,
    })
    e.update({k: str(v) for k, v in extra.items()})
    return e


if os.environ.get("FASTWAM_REDIRECT_COMMON_FILES", "false").lower() == "true":
    _wan_files = [
        os.path.join(DIFFSYNTH, "DiffSynth-Studio", "Wan-Series-Converted-Safetensors", "models_t5_umt5-xxl-enc-bf16.safetensors"),
        os.path.join(DIFFSYNTH, "DiffSynth-Studio", "Wan-Series-Converted-Safetensors", "Wan2.2_VAE.safetensors"),
    ]
else:
    _wan_files = [
        os.path.join(DIFFSYNTH, "Wan-AI", "Wan2.2-TI2V-5B", "models_t5_umt5-xxl-enc-bf16.pth"),
        os.path.join(DIFFSYNTH, "Wan-AI", "Wan2.2-TI2V-5B", "Wan2.2_VAE.pth"),
    ]
_required = [
    CKPT,
    DATASET_STATS,
    *_wan_files,
    os.path.join(DIFFSYNTH, "Wan-AI", "Wan2.1-T2V-1.3B", "google", "umt5-xxl", "tokenizer.json"),
]
_missing = [p for p in _required if not os.path.exists(p)]
if _missing:
    raise FileNotFoundError("FastWAM asset preparation incomplete:\n  " + "\n  ".join(_missing))
print("FastWAM public assets ready")
print("  checkpoint    :", CKPT)
print("  dataset stats :", DATASET_STATS)
print("  Wan components:", DIFFSYNTH)
print("  GT episodes   :", DATA_DIR if DOWNLOAD_DATASET else "skipped (interactive-only mode)")
print("  interpreter   :", FASTWAM_PY)

## 1. Interactive LIBERO simulator (fast action route)

Run the cell below, wait for the model to load (the first run JIT-compiles GPU kernels - it can take
a couple of minutes), then the live sim appears inline. Pick a task from the **environment**
dropdown, type an instruction, and press **Send**. The status line shows the step count and the
live **per-plan latency** (the fast `infer_action` time). This is the default, fastest route.

In [ ]:
from IPython.display import IFrame, display

# The sim server binds 0.0.0.0:RT_PORT. Under Ryzers the container shares the host network
# (--network=host), so the IFrame below loads http://localhost:RT_PORT/ directly. The
# JUPYTERHUB_SERVICE_PREFIX branch further down is kept for the JupyterHub deployment,
# where the same UI is proxied instead.
RT_PORT = os.environ.get("RT_PORT", "8080")
SIM_SERVER = "/ryzers/notebooks/scripts/interactive_server_fastwam.py"

# Stop a server started by a previous run of this cell.
try:
    if globals().get("_sim") and _sim.poll() is None:
        _sim.terminate()
        _sim.wait(timeout=5)
except Exception:
    pass

# Serve the FastWAM policy on the fast action route. Stream server logs to a file so the notebook
# kernel never blocks on a full stdout pipe while the sim runs.
sim_env = child_env(
    PORT=RT_PORT,
    SUITE=os.environ.get("SUITE", "libero_object"),
    TASK_ID=os.environ.get("TASK_ID", "0"),
)
_log_path = os.path.join(OUT_DIR, "interactive_fastwam.log")
_log = open(_log_path, "w")
_sim = subprocess.Popen([FASTWAM_PY, SIM_SERVER], env=sim_env, stdout=_log, stderr=subprocess.STDOUT)
print(f"FastWAM sim server started (pid {_sim.pid}); loading T5 + VAE + DiT ...")
print("(first run JIT-compiles ROCm kernels; this can take a few minutes)")


def _sim_status():
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{RT_PORT}/status", timeout=3) as r:
            return json.load(r)
    except Exception:
        return None


ready, deadline = False, time.time() + 1200
while time.time() < deadline:
    if _sim.poll() is not None:
        print("\nserver exited early; tail of log:")
        print("".join(open(_log_path).readlines()[-25:]))
        break
    s = _sim_status()
    if s:
        print("  " + str(s.get("status", ""))[:96], end="\r")
        if s.get("mode") in ("idle", "running"):
            ready = True
            break
        if s.get("mode") == "error":
            print("\nserver error:", s.get("status"))
            print("".join(open(_log_path).readlines()[-25:]))
            break
    time.sleep(3)

prefix = os.environ.get("JUPYTERHUB_SERVICE_PREFIX", "").rstrip("/")
sim_url = f"{prefix}/proxy/{RT_PORT}/" if prefix else f"http://localhost:{RT_PORT}/"
if ready:
    print(f"\nready - interactive FastWAM sim embedded below (also open in a tab: {sim_url})")
    display(IFrame(sim_url, width="100%", height=860))
else:
    print(f"\nserver not ready yet; wait a moment and re-run this cell. URL: {sim_url}")

## 2. Video imagination - what the model "dreams"

FastWAM is a **world** model, so it can imagine the *future video*, not just the next actions. The
cell below imagines **two full episodes from two different LIBERO-Object tasks** - the *bbq sauce*
pick and the *milk* pick - and shows **ground truth (left) vs FastWAM's imagination (right)** side
by side, for the entire episode.

To cover a whole episode (not just one 32-step chunk) it runs `infer_joint` as a **receding-horizon
rollout**: each call imagines the next horizon of video+actions from the real observation at that
step, and the segments are stitched into one full-length clip - the same receding-horizon loop the
shipped LIBERO eval uses.

This is the heavier route (a full video diffusion per horizon), so the cell first **stops the live
sim above** to free the GPU, and the **first horizon JIT-compiles ROCm kernels** (a few minutes;
each subsequent horizon is far faster). The setup cell downloads the public
`yuanty/LIBERO-fastwam` replay archive used here. Knobs: `EPISODES="47,273"` picks the two episodes
(any episode indices; different tasks live at different indices), `NUM_STEPS` trades speed for
quality, and `MAX_STEPS_PER_EPISODE` caps length if you want a quicker preview.

> **Note:** Video imagination is the heavy route - a full video diffusion per horizon - so the cell below first stops the live sim from section 1 to give the GPU its full memory back. Both routes would otherwise each hold a full model copy in memory at once.

In [ ]:
from IPython.display import Video, HTML, display

# Free the GPU: stop the interactive sim (both would otherwise hold a full model copy in memory).
try:
    if globals().get("_sim") and _sim.poll() is None:
        print("stopping the live sim to free the GPU for imagination ...")
        _sim.terminate()
        _sim.wait(timeout=10)
except Exception:
    pass

VIDEOGEN = "/ryzers/notebooks/scripts/fastwam_videogen.py"
# Two DIFFERENT tasks, rendered as FULL episodes (receding-horizon rollouts, not a single chunk):
# ep47 = "pick up the bbq sauce...", ep273 = "pick up the milk...". Override via EPISODES="e0,e1".
gen_env = child_env(
    EPISODES=os.environ.get("EPISODES", "47,273"),
    NUM_STEPS=os.environ.get("NUM_STEPS", "20"),
    TAG=os.environ.get("SUITE", "libero_object"),
)
print("imagining FULL episodes for two tasks (loads the model, then runs the joint video+action")
print("route as a receding-horizon rollout over each whole episode)...")
print("the first horizon JIT-warms ROCm kernels, so this takes a few minutes.\n")

# Stream the backend's output live and pick up the machine-parseable lines it prints per clip.
proc = subprocess.Popen([FASTWAM_PY, VIDEOGEN], env=gen_env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
mp4s, summary = [], None
for line in proc.stdout:
    line = line.rstrip()
    if line.startswith("VIDEOGEN_MP4="):
        mp4s.append(line.split("=", 1)[1])
    elif line.startswith("VIDEOGEN_SUMMARY="):
        try:
            summary = json.loads(line.split("=", 1)[1])
        except Exception:
            pass
    else:
        print(line)
proc.wait()

clip_task = {c["mp4"]: c for c in (summary.get("clips", []) if summary else [])}
if summary:
    print(f"\njoint-path latency ({summary['num_inference_steps']} steps): first horizon "
          f"{summary['joint_latency_s_first_warmup']:.1f}s (JIT warmup), then "
          f"~{summary['joint_latency_s_mean_steady']:.1f}s per horizon")
for i, mp4 in enumerate(mp4s):
    if os.path.exists(mp4):
        c = clip_task.get(mp4, {})
        task = c.get("task", f"clip {i}")
        meta = f" &mdash; {c['frames']} frames, {c['windows']} horizons" if c else ""
        display(HTML(f"<b>Full episode &mdash; {task}</b><br>"
                     f"<span>ground truth (left) vs FastWAM imagined (right){meta}</span>"))
        display(Video(mp4, embed=True, width=900))
    else:
        print("missing clip:", mp4)
if not mp4s:
    print("no clips produced - check the log above.")

## Key Takeaways

Now you know:
- What a **world-action model** is: FastWAM can plan actions on a fast route *and* imagine the future video on its joint route
- How the same **receding-horizon** loop drives both the live interactive sim and the full-episode imagination rollouts
- Why the heavier video-imagination route frees the GPU first, and that the first horizon pays a one-time JIT-compile cost
- How a GPU-rendered sim runs inline from the container's host network in its own isolated ROCm venv

## References

* [FastWAM code](https://github.com/yuantianyuan01/FastWAM)
* [FastWAM checkpoint (Hugging Face)](https://huggingface.co/yuanty/fastwam)
* [FastWAM LIBERO replay data (Hugging Face)](https://huggingface.co/datasets/yuanty/LIBERO-fastwam)
* [Wan2.2-TI2V-5B](https://huggingface.co/Wan-AI/Wan2.2-TI2V-5B)

FastWAM is built on the **Wan2.2-TI2V-5B** video diffusion model (a T5 text encoder, a Wan video VAE, and a video/action diffusion transformer).

**Back to**: [0_overview.ipynb](0_overview.ipynb)

---
Copyright© 2026 AMD, Inc SPDX-License-Identifier: MIT